# Notebook 08 — Time-Series Forecasting with a Transformer

## Learning Objectives
- Apply Transformers to a non-NLP problem: univariate time-series forecasting.
- Load synthetic sine-wave data using `src.data.synthetic_timeseries`.
- Build and train a `TimeSeriesTransformer` (encoder-only, regression head).
- Visualize forecasts: past context, true future, and predicted future on the same plot.
- Compare training MSE and MAE over epochs.

## Big Picture
Transformers are not limited to text or images. Any sequence of vectors can be
processed by the encoder's attention mechanism. In time-series forecasting:
- Each **time step** is analogous to a **token**.
- The **scalar value** at each step is projected to d_model (analogous to embedding).
- We predict the next `forecast_length` steps from the past `input_length` steps.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report
from src.utils.paths import RUNS_TIMESERIES_DIR
from src.data.synthetic_timeseries import load_timeseries_data, SyntheticTimeSeriesDataset
from src.models.time_series_transformer import TimeSeriesTransformer
from src.training.timeseries import timeseries_loss_fn, timeseries_metric_fn

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_TIMESERIES_DIR, clean=True)
print(f"Device  : {device}")
print(f"Run dir : {RUNS_TIMESERIES_DIR}")

## 2. Dataset: Synthetic Sine-Wave Time Series

We generate a noisy sine wave with multiple frequencies.
Each sample is a sliding window: given 48 past steps, predict the next 12 steps.

This is a standard **multi-step forecasting** setup used in many real applications
(energy, weather, stock prices, traffic).

In [ ]:
# Hyperparameters
INPUT_LENGTH    = 48
FORECAST_LENGTH = 12
D_MODEL         = 64
NUM_HEADS       = 4
NUM_LAYERS      = 2
DIM_FF          = 128
BATCH_SIZE      = 32
NUM_EPOCHS      = 8
LR              = 1e-3

# Load data
train_loader, val_loader = load_timeseries_data(
    n_train=800,
    n_val=200,
    input_length=INPUT_LENGTH,
    forecast_length=FORECAST_LENGTH,
    mode='noisy',
    noise_std=0.1,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

# Inspect one batch
x_batch, y_batch = next(iter(train_loader))
print(f"Input  (past)   shape: {x_batch.shape}  (batch, input_length, 1)")
print(f"Target (future) shape: {y_batch.shape}  (batch, forecast_length, 1)")
print(f"\nInput value range : [{x_batch.min().item():.3f}, {x_batch.max().item():.3f}]")
print(f"Target value range: [{y_batch.min().item():.3f}, {y_batch.max().item():.3f}]")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
# Visualize a few raw time-series samples
fig, axes = plt.subplots(1, 3, figsize=(15, 3), sharey=True)
for i, ax in enumerate(axes):
    past   = x_batch[i, :, 0].numpy()     # [input_length]
    future = y_batch[i, :, 0].numpy()     # [forecast_length]
    t_past   = np.arange(INPUT_LENGTH)
    t_future = np.arange(INPUT_LENGTH, INPUT_LENGTH + FORECAST_LENGTH)

    ax.plot(t_past,   past,   color='steelblue', linewidth=1.5, label='Past (input)')
    ax.plot(t_future, future, color='tomato',    linewidth=1.5, label='True future')
    ax.axvline(INPUT_LENGTH, color='black', linestyle='--', alpha=0.5)
    ax.set_title(f'Sample {i}')
    if i == 0:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Synthetic Noisy Time Series (input={INPUT_LENGTH}, forecast={FORECAST_LENGTH})', fontsize=12)
fig.tight_layout()
raw_path = RUNS_TIMESERIES_DIR / 'raw_samples.png'
fig.savefig(raw_path, dpi=120)
plt.close(fig)
print(f"Raw sample plot saved to: {raw_path}")

## 3. Architecture (Text Diagram)

```
Input: [batch, input_length=48, 1]   (one scalar per time step)
        │
  Linear(1 → d_model=64)             (analogous to token embedding)
  [batch, 48, 64]
        │
  + SinusoidalPositionalEncoding     (tells model which time step this is)
        │
  TransformerEncoder (2 blocks)
  [batch, 48, 64]
        │
  Mean Pool → [batch, 64]
        │
  Regression Head: Linear(64→128) → GELU → Linear(128→12)
  [batch, forecast_length=12]
        │
  Unsqueeze → [batch, 12, 1]   (add channel dim)
        │
  MSE Loss vs true future values
```

## 4. Theory: Transformers for Time Series

**Why attention works for time series**:
- Long-range dependencies (e.g., weekly seasonality in daily data) are handled naturally.
- The model can learn to attend to similar past patterns ("last Monday looked like this too").
- No vanishing gradient issues with attention (unlike RNNs for long sequences).

**Input projection** replaces the token embedding: since our input is a scalar (1 feature),
we use a learned linear layer to project it to d_model dimensions.

**Regression head**: Instead of class logits, we predict future values directly.
We use MSE (mean squared error) as the loss function — measures average squared deviation.

$$\mathcal{L}_{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2$$

## 5. Implementation from Scratch

In [ ]:
# Build the model
model = TimeSeriesTransformer(
    input_size=1,
    forecast_length=FORECAST_LENGTH,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    max_len=INPUT_LENGTH + 8,
    dropout=0.1,
).to(device)

n_params = model.count_parameters()
print(f"Trainable parameters: {n_params:,}")

# Shape trace
x_sample = x_batch[:4].to(device)   # [4, 48, 1]
with torch.no_grad():
    forecast = model(x_sample)       # [4, 12, 1]

print(f"\nInput  shape   : {x_sample.shape}   (batch, input_length, 1)")
print(f"Forecast shape : {forecast.shape}  (batch, forecast_length, 1)")
print(f"Forecast range : [{forecast.min().item():.4f}, {forecast.max().item():.4f}]")

## 6. Sanity Checks

In [ ]:
# Shape checks
assert forecast.shape == (4, FORECAST_LENGTH, 1), f"Unexpected shape: {forecast.shape}"
print("Forecast shape check passed.")

# No NaN in output
assert not torch.isnan(forecast).any(), "NaN in forecast output!"
print("No NaN in output.")

# Initial MSE should be O(1) — data is normalized to ~[-1, 1]
init_mse = F.mse_loss(forecast, y_batch[:4].to(device)).item()
print(f"Initial MSE: {init_mse:.4f}  (expected O(1) with random init)")

# Baseline: always predict zero (mean of normalized signal)
zero_pred = torch.zeros_like(y_batch[:4])
zero_mse  = F.mse_loss(zero_pred, y_batch[:4]).item()
print(f"Zero-prediction MSE baseline: {zero_mse:.4f}")
print("All sanity checks passed!")

## 7. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_mse_list, val_mse_list = [], []
train_mae_list, val_mae_list = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    model.train()
    ep_mse, ep_mae, n_batches = 0.0, 0.0, 0
    for batch in train_loader:
        x, y = batch
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)                          # [batch, forecast_length, 1]
        loss = F.mse_loss(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_mse += loss.item()
        ep_mae += F.l1_loss(pred.detach(), y).item()
        n_batches += 1
    train_mse_list.append(ep_mse / max(n_batches, 1))
    train_mae_list.append(ep_mae / max(n_batches, 1))

    # ── Validate ──
    model.eval()
    v_mse, v_mae, v_batches = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x, y = x.to(device), y.to(device)
            pred = model(x)
            v_mse += F.mse_loss(pred, y).item()
            v_mae += F.l1_loss(pred, y).item()
            v_batches += 1
    val_mse_list.append(v_mse / max(v_batches, 1))
    val_mae_list.append(v_mae / max(v_batches, 1))

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}  "
          f"train_mse={train_mse_list[-1]:.4f}  train_mae={train_mae_list[-1]:.4f}  "
          f"val_mse={val_mse_list[-1]:.4f}  val_mae={val_mae_list[-1]:.4f}")

print("Training complete!")

## 8. Evaluation

In [ ]:
# Final evaluation on full validation set
model.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        pred = model(x).cpu()
        all_preds.append(pred)
        all_trues.append(y)

all_preds = torch.cat(all_preds)   # [n_val, forecast_length, 1]
all_trues = torch.cat(all_trues)   # [n_val, forecast_length, 1]

final_mse = F.mse_loss(all_preds, all_trues).item()
final_mae = F.l1_loss(all_preds,  all_trues).item()

# Baseline: always predict the last observed value
# (naive forecast: last value of input carried forward)

print(f"Final validation MSE : {final_mse:.4f}")
print(f"Final validation MAE : {final_mae:.4f}")
print(f"Zero-pred MSE (baseline): {zero_mse:.4f}")
print(f"Improvement over zero-pred: {100*(1 - final_mse/zero_mse):.1f}%")

# Save metrics
metrics = {
    'final_val_mse': final_mse,
    'final_val_mae': final_mae,
    'final_train_mse': train_mse_list[-1],
    'num_epochs': NUM_EPOCHS,
    'input_length': INPUT_LENGTH,
    'forecast_length': FORECAST_LENGTH,
    'num_params': n_params,
}
save_metrics_json(metrics, RUNS_TIMESERIES_DIR / 'metrics.json')
print(f"\nMetrics saved to: {RUNS_TIMESERIES_DIR / 'metrics.json'}")

## 9. Visualization

In [ ]:
# Figure 1: Training curves (MSE and MAE)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, NUM_EPOCHS + 1)

ax1.plot(ep, train_mse_list, 'o-', markersize=4, label='Train MSE', color='steelblue')
ax1.plot(ep, val_mse_list,   's-', markersize=4, label='Val MSE',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title('Mean Squared Error'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep, train_mae_list, 'o-', markersize=4, label='Train MAE', color='steelblue')
ax2.plot(ep, val_mae_list,   's-', markersize=4, label='Val MAE',   color='tomato')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE')
ax2.set_title('Mean Absolute Error'); ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('TimeSeriesTransformer Training Curves', fontsize=12)
fig.tight_layout()
curve_path = RUNS_TIMESERIES_DIR / 'training_curve.png'
fig.savefig(curve_path, dpi=120)
plt.close(fig)
print(f"Training curves saved to: {curve_path}")

In [ ]:
# Figure 2: Forecast plots — past / true future / predicted future
n_examples = 4
fig, axes = plt.subplots(1, n_examples, figsize=(16, 3.5), sharey=True)

model.eval()
# Get fresh examples from the val loader
x_vis, y_vis = next(iter(val_loader))

with torch.no_grad():
    pred_vis = model(x_vis[:n_examples].to(device)).cpu()  # [n_examples, 12, 1]

for i, ax in enumerate(axes):
    past      = x_vis[i, :, 0].numpy()        # [input_length]
    true_fut  = y_vis[i, :, 0].numpy()        # [forecast_length]
    pred_fut  = pred_vis[i, :, 0].numpy()     # [forecast_length]

    t_past   = np.arange(INPUT_LENGTH)
    t_future = np.arange(INPUT_LENGTH, INPUT_LENGTH + FORECAST_LENGTH)

    ax.plot(t_past,   past,      color='steelblue', linewidth=1.5, label='Past input', alpha=0.8)
    ax.plot(t_future, true_fut,  color='green',     linewidth=2,   label='True future', linestyle='--')
    ax.plot(t_future, pred_fut,  color='tomato',    linewidth=2,   label='Predicted',   linestyle='-')
    ax.axvline(INPUT_LENGTH, color='black', linestyle=':', alpha=0.5)

    sample_mse = ((pred_fut - true_fut)**2).mean()
    ax.set_title(f'Sample {i}\nMSE={sample_mse:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')
    ax.set_xlabel('Time step')

axes[0].set_ylabel('Value')
fig.suptitle(
    f'Time-Series Forecasting: Past ({INPUT_LENGTH} steps) → Predicted ({FORECAST_LENGTH} steps)',
    fontsize=12
)
fig.tight_layout()
forecast_path = RUNS_TIMESERIES_DIR / 'forecast_plot.png'
fig.savefig(forecast_path, dpi=120)
plt.close(fig)
print(f"Forecast plot saved to: {forecast_path}")

In [ ]:
# Save session report
save_markdown_report(
    title='Time-Series Forecasting',
    summary=(
        f'TimeSeriesTransformer trained for {NUM_EPOCHS} epochs on synthetic noisy sine waves. '
        f'Final val MSE: {final_mse:.4f}, MAE: {final_mae:.4f}.'
    ),
    metrics=metrics,
    figures=['training_curve.png', 'forecast_plot.png', 'raw_samples.png'],
    tables=[],
    output_path=RUNS_TIMESERIES_DIR / 'session_report.md',
    device=str(device),
    hyperparams={
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'input_length': INPUT_LENGTH,
        'forecast_length': FORECAST_LENGTH,
        'lr': LR,
        'batch_size': BATCH_SIZE,
    },
    architecture=(
        'TimeSeriesTransformer: Linear(1→d_model) + SinPE + TransformerEncoder×2 '
        '+ mean_pool + MLP(d_model→forecast_length)'
    ),
    loss_fn='MSELoss',
)
print(f"Session report saved to: {RUNS_TIMESERIES_DIR / 'session_report.md'}")

## 10. Failure Cases

**Model predicts a flat line**: This indicates the regression head learned to always predict zero (the mean of the normalized signal). Check that the MSE is actually decreasing past zero_mse. If not, try a lower learning rate or more epochs.

**Overfitting on noisy signal**: A model too large will fit the training noise and generalize poorly. Signs: val_mse >> train_mse. Fix: reduce d_model/layers or increase dropout.

**Bad forecast at boundary**: The first few forecast steps are usually more accurate than later steps because error accumulates (even though our model makes all predictions at once, the signal-to-noise ratio degrades for later steps).

**Input not normalized**: Without normalization, large-magnitude signals cause instability. The `SyntheticTimeSeriesDataset` already normalizes to ~[-1, 1]. Always normalize real-world data.

## 11. Exercises

1. **Longer forecast**: Increase `FORECAST_LENGTH` to 24. How does MAE change? At what horizon does the model become worse than naive (zero-prediction)?
2. **Multi-frequency signal**: Change `mode='multi_freq'`. Does the Transformer still forecast well? Compare MSE with `mode='clean'`.
3. **Input projection ablation**: Replace the `Linear(1→d_model)` with a fixed (non-learned) projection. Does the model still learn? Why or why not?
4. **Attention visualization**: Extract the encoder attention weights for a single sample. Which past time steps does the model attend to most when making its prediction?
5. **Compare to LSTM**: Implement a 2-layer LSTM forecaster with the same parameter budget. Which achieves lower val MSE in 8 epochs?

## 12. Key Takeaways

- **Transformers generalize beyond NLP**: any sequence of vectors can be processed by the encoder.
- **Input projection** replaces token embeddings for continuous-valued time series.
- **Regression head** (MSE loss) replaces the classification head (cross-entropy loss).
- **Mean pooling** summarizes the full input context; attention can pick out relevant past patterns.
- Normalizing the input signal to ~[-1, 1] is critical for training stability.
- The same architecture (encoder + pool + MLP) works for classification, regression, and forecasting — only the head changes.